In [1]:
import numpy as np
import warnings
from data.style_components.matplotlib_style import *
%matplotlib notebook

$$    S(t, \omega) =  2\cdot \int_{-\infty}^{\infty}\mathrm{d\nu} \hspace{1mm} \tilde{\xi}^{\ast}(\omega + \nu) \tilde{\xi}(\omega-\nu)e^{-4\pi i\nu t}.$$

Discretized version:

$$        S_{ij}:=S(t_i, \omega_j) = 2\cdot \Delta\nu\sum_{l=l_{\mathrm{min}}}^{l_{\mathrm{max}}} \hspace{1mm} \big(\tilde{\xi}^{\ast}\big)^{j + l} \tilde{\xi}^{j-l}e^{-4\pi i\nu_l t_i}.$$

In [66]:
def sn_complex_xi(size, scale=1):
    # standard normal, complex valued xi
    # scale sets the standard deviation! => scale' = sqrt(scale) if you want to set variance by scale
    res = 1/2*np.random.randn(size) + 1/2*np.random.randn(size) * 1j
    return scale * res

def raise_warning(msg):
    warnings.warn(msg, category=UserWarning, stacklevel=2)

In [44]:
def stress_field(complex_xi, L, N):
    """
    Calculates the stress field in matrix representation. Calculates the stress field in matrix representation
    :param complex_xi:      The complex xi field to analyze.
    :param L:               The length of the real space domain.
    :param N:               The number of support/sampling points in real space.
    :return:
    """
    delta_nu = 1/L
    delta_t = L/N
    complex_xi_conj = np.conj(complex_xi)

    S_mat = np.zeros((N, N), dtype=complex)

    for i in range(N):
        # fixes t
        for j in range(N):
            # fixes omega
            to_sum = []
            l_min = int(max(-N/2, -j, j-N+1))
            l_max = int(min(N/2, N-1-j, j))
            for l in range(l_min, l_max):
                nu_l = delta_nu * l
                t_i = delta_t * i
                phase = np.exp(-4*np.pi * 1j * nu_l * t_i)
                res = complex_xi_conj[j+l] * complex_xi[j-l] * phase
                to_sum.append(res)

            S_mat[i,j] = delta_nu * np.sum(to_sum)

    S_mat *= 2
    diagnostic = np.mean(S_mat.imag)
    if diagnostic < 1e-10:
        print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
    else:
        raise_warning(f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")
    return S_mat


In [45]:
n_pix = 200
length = 2
rnd_xi = sn_complex_xi(n_pix)

In [46]:
S = stress_field(rnd_xi, L=length, N=n_pix)

✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-4.04121180963557e-18) 


In [47]:
print(np.mean(S.imag), " ~ 0 so effectively real as expected...")

-4.04121180963557e-18  ~ 0 so effectively real as expected...


Let's now check a $\tilde{\xi}$ field that represents a very strong, extremely localized stress point at the frequency $\omega_{10} := 10 * \Delta \nu$:

$$\tilde{\xi}^j =\delta_{j, 10} a$$

with some very large number $a$.

In [48]:
a = 1e2
xi_peak = np.ones_like(rnd_xi)*0
xi_peak[10] = a

In [49]:
S_delta_peak = stress_field(xi_peak, L=length, N=n_pix)

✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (0.0) 


In [50]:
fig1 = plt.figure(figsize=(6,5))
plt.imshow(S_delta_peak.real, origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(label='Re(S)')
plt.xlabel('Frequency index (j)')
plt.ylabel('Time index (i)')
plt.title('Re(Stress Field S)')
plt.tight_layout()
plt.show()

<IPython.core.display.Javascript object>

What if we actually tested a field that has a constant elevation besides that one peak in frequency space? Something like :

In [51]:
b = 100
xi_peak_constant_elevation = np.ones(len(rnd_xi))*5
xi_peak_constant_elevation[10] = b
fig2 = plt.figure(figsize=(6,5))
plt.plot(xi_peak_constant_elevation)
plt.xlabel('Frequency index (j)')
plt.ylabel(r'Harmonic field $\xi$')
# plt.ylim(0, 11)
plt.tight_layout()
plt.show()

<IPython.core.display.Javascript object>

In [52]:
S_delta_peak_cst_elevation = stress_field(xi_peak_constant_elevation, L=length, N=n_pix)

✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (7.105427357601002e-16) 


In [53]:
fig3 = plt.figure(figsize=(6,5))
plt.imshow(S_delta_peak_cst_elevation.real, origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(label='Re(S)')
plt.xlabel('Frequency index (j)')
plt.ylabel('Time index (i)')
plt.title('Re(Stress Field S)')
plt.tight_layout()
plt.show()

<IPython.core.display.Javascript object>

Don't know how to interpret these stripes... Also raised stress levels at $t=0$ line??

Now, in our continuous formula, an average over multiple $\xi$'s that are from $\mathcal{G}(0,1)$, should give a flat color image.


In [67]:
S_mat_collection = []
for _ in range(10):
    rnd_xi = sn_complex_xi(n_pix, scale=np.sqrt(length))
    S_mat = stress_field(rnd_xi, L=length, N=n_pix)
    S_mat_collection.append(S_mat)

✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-1.7328360968349443e-16) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-6.821210263296961e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (6.172840016915871e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-3.730349362740526e-18) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-2.0250467969162856e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.1937117960769682e-16) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (6.235012506294879e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (6.80344669490296e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (-1.4388490399142028e-17) 
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (9.183764859699295e-17) 


In [68]:
average_S_mat = np.mean(S_mat_collection, axis=0)

In [69]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# First subplot: first entry of S_mat_collection
im0 = axes[0].imshow(S_mat_collection[0].real, origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(im0, ax=axes[0], label='Re(S)')
axes[0].set_xlabel('Frequency index (j)')
axes[0].set_ylabel('Time index (i)')
axes[0].set_title('Re(S) – First Entry')

# Second subplot: average S_mat
im1 = axes[1].imshow(average_S_mat.real, origin='lower', aspect='auto', cmap='viridis')
plt.colorbar(im1, ax=axes[1], label='Re(S)')
axes[1].set_xlabel('Frequency index (j)')
axes[1].set_ylabel('Time index (i)')
axes[1].set_title('Re(S) – Average')

plt.tight_layout()
plt.show()

print("Average stress, std on the left: ",np.round(np.mean(S_mat_collection[0].real),2), np.round(np.std(S_mat_collection[0].real),2),  " Average stress, std on the right: ", np.round(np.mean(average_S_mat.real),2), np.round(np.std(average_S_mat.real),2))

<IPython.core.display.Javascript object>

Average stress, std on the left:  0.98 9.82  Average stress, std on the right:  0.98 3.12


We see that, against our expectations, the average over multiple such fields is not constant $=1$ over all pairs of $(t, \omega)$. Instead, the mean seems to be $0$ (which is also nice to say that the stress is $0$ in the simplest case) with relatively small dispersion.

Therefore, it looks like I have made some mistake in the discretization procedure?

UPDATE: Ok so now this completely works. The question is: If we just saw this sample on the left, would we have detected an anomaly somewhere? The mean is hovering around where it should, but the variance is quite large, i.e. there exist small pixels with a lot of variance. The key to anomaly detection here is likely the homogeneity of the picture: I expect that if we plot the distribution of the color values, we would get a Gaussian with variance 1 in both directions...

In [85]:
fig4 = plt.figure(figsize=(6,5))
helper = S_mat_collection[0].real.flatten()
counts_bins = plt.hist(helper, bins=20, density=True)
counts, bins = counts_bins[0], counts_bins[1]
bins_explicit = np.linspace(-48.79233786, 48.4606015, 100)
import scipy
y = scipy.stats.norm.pdf(bins_explicit, loc=np.mean(helper), scale=np.std(helper))
plt.plot(bins_explicit, y)

plt.show()

<IPython.core.display.Javascript object>